# Project 1: Build an LLM Playground

Welcome to your first project! In this project, you'll build a simple large language model (LLM) playground, an interactive environment where you can experiment with LLMs and understand how they work under the hood.

The goal here is to understand the foundations and mechanics behind LLMs rather than relying on higher-level abstractions or frameworks. You'll see what happens “under the hood”, how an LLM receives a text, processes it, and generate a response. In later projects, you'll use frameworks like Ollama and LangChain that simplify many of these steps. But before that, this project will help you build a solid mental model of how LLMs actually work.

We'll use Google Colab, a free browser-based platform that lets you run Python code and machine learning models without installing anything locally. Click the button below to open this notebook in Colab.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bytebyteai/ai-eng-projects-2/blob/main/project_1/lm_playground.ipynb)

If you prefer to run the project locally, you can use the provided `env.yaml` file to create a compatible environment using conda. To do so, open a terminal in the same directory as this notebook and run:

```bash
# Create and activate the conda environment
conda env create -f env.yaml && conda activate llm_playground

# Register this environment as a Jupyter kernel
python -m ipykernel install --user --name=llm_playground --display-name "llm_playground"
```


---
## Learning Objectives  
- Understand tokenization and how raw text is converted into a sequence of discrete tokens
- Inspect GPT-2 and the Transformer architecture
- Learn how to load pretrained LLMs using Hugging Face
- Explore decoding strategies to generate text from LLMs
- Compare completion models with instruction-tuned models


Let's get started!

In [1]:
# Confirm required libraries are installed and working.
import torch, transformers, tiktoken
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("✅ Environment check complete. You're good to go!")

torch 2.9.0+cpu | transformers 4.57.1
✅ Environment check complete. You're good to go!


# 1 - Tokenization

A neural network cannot process raw text directly. It needs numbers.
Tokenization is the process of converting text into numerical IDs that models can understand. In this section, you will learn how tokenization works in practice and why it is an essential step in every language model pipeline.

Tokenization methods generally fall into three main categories:
1. Word-level
2. Character-level
3. Subword-level

### 1.1 - Word-level tokenization
This method splits text by whitespace and treats each word as a single token. In the next cell, you will implement a basic word-level tokenizer by building a vocabulary that maps words to IDs and writing `encode` and `decode` functions.

In [2]:
# Creating a tiny corpus. In practice, a corpus is generally the entire internet-scale dataset used for training.
corpus = [
    "The quick brown fox jumps over the lazy dog",
    "Tokenization converts text to numbers",
    "Large language models predict the next token"
]

# Step 1: Build vocabulary (all unique words in the corpus) and mappings
vocab = []
word2id = {}
id2word = {}

# Split the sentence into words
words = []

for sentence in corpus:
    words.extend(sentence.split())
    
print(words)

# Get only unique words (no repeats)
for w in words:
    if w not in vocab:
        vocab.append(w)     #vocab = list(set(corpus.split()))

for i, word in enumerate(vocab):              #mapping of word to id
    word2id[word] = i

print(word2id)

for word, i in word2id.items():               #mapping of id to word
    id2word[i] = word

print(id2word)

vocab.append("<UNK>")
word2id["<UNK>"] = len(word2id)
id2word[word2id["<UNK>"]] = "<UNK>"

print(f"Vocabulary size: {len(vocab)} words")
print("First 15 vocab entries:", vocab[:15])


['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'Tokenization', 'converts', 'text', 'to', 'numbers', 'Large', 'language', 'models', 'predict', 'the', 'next', 'token']
{'The': 0, 'quick': 1, 'brown': 2, 'fox': 3, 'jumps': 4, 'over': 5, 'the': 6, 'lazy': 7, 'dog': 8, 'Tokenization': 9, 'converts': 10, 'text': 11, 'to': 12, 'numbers': 13, 'Large': 14, 'language': 15, 'models': 16, 'predict': 17, 'next': 18, 'token': 19}
{0: 'The', 1: 'quick', 2: 'brown', 3: 'fox', 4: 'jumps', 5: 'over', 6: 'the', 7: 'lazy', 8: 'dog', 9: 'Tokenization', 10: 'converts', 11: 'text', 12: 'to', 13: 'numbers', 14: 'Large', 15: 'language', 16: 'models', 17: 'predict', 18: 'next', 19: 'token'}
Vocabulary size: 21 words
First 15 vocab entries: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'Tokenization', 'converts', 'text', 'to', 'numbers', 'Large']


In [3]:
# Step 2: Define encode and decode functions
def encode(text):
    # converts text to token IDs
    words = text.split()
    token_ids = []
    for w in words:
        if w in word2id:
            token_ids.append(word2id[w])
        else:
            token_ids.append(word2id["<UNK>"])
    return token_ids


def decode(ids):
    # converts token IDs back to text
    words = []
    for tokenId in ids:
        words.append(id2word[tokenId])
    return " ".join(words)    

In [6]:
# Step 3: Test your tokenizer with random sentences. 
# Try a sentence with unseen words and see what happens (and how to fix it)

sentence = "hello rajdeep"
encoded = encode(sentence)
print(f"encoded: {encoded}")


encoded: [20, 20]


While word-level tokenization is simple and easy to understand, it has two key limitations that make it impractical for large-scale models:
1.  large vocabulary size: every new word or variation (for example, run, runs, running) increases the total vocabulary, leading to higher memory and training costs.
2. Out-of-vocabulary (OOV) problem: the model cannot handle unseen or rare words that were not part of the training vocabulary, so they must be replaced with a generic [UNK] token.

The next section introduces character-level tokenization, where text is represented as individual characters instead of words.

### 1.2 - Character-level tokenization

In this approach, every single character (including spaces, punctuation, and even emojis) is assigned its own ID.

In the next section, we will rebuild a tokenizer using the same corpus as before, but this time with a character-level approach.
For simplicity, assume we are only using lowercase and uppercase English letters (a-z, A-Z).

In [7]:
import string


# Step 1: Create a vocabulary that includes all uppercase and lowercase letters.

vocab = []
char2id = {}
id2char = {}

vocab = list(string.ascii_letters)
vocab.append("<UNK>")

#create mappings

for i, char in enumerate(vocab):              #mapping of word to id
    char2id[char] = i
             
for char, i in enumerate(char2id.items()):
    id2char[i] = char

# There is an alternative way to map

corpus = [
    "The quick brown fox jumps over the lazy dog",
    "Tokenization converts text to numbers",
    "Large language models predict the next token"
]
words = []

for sentence in corpus:
    words.extend(sentence.split())

# created vocab2 from existing corpus

# vocab2 = []

# for word in words:
#     for char in word:
#         if char not in vocab2:
#             vocab2.append(char)
# print("vocab2 from corpus " + vocab2)

vocab.append("<UNK>")
char2id["<UNK>"] = len(char2id)
id2char[char2id["<UNK>"]] = "<UNK>"
    

print(f"Vocabulary size: {len(vocab)} (52 letters + 2 specials)")


Vocabulary size: 54 (52 letters + 2 specials)


In [8]:
# Step 2: Implement encode() and decode() functions to convert between text and IDs.
def encode(text):
    # convert text to list of IDs

    token_ids = []
    for char in text:
        if char in char2id:
            token_ids.append(char2id[char])
        else:
            token_ids.append(char2id["<UNK>"])
    return token_ids 


def decode(ids):
    # Convert list of IDs to text
    characters = []
    for tokenId in ids:
        characters.append(id2char[tokenId])
    return " ".join(characters)  
    
    
    

In [9]:
# Step 3: Test your tokenizer on a short sample word.
sentence = "Dinosaurs are Extinct"
encoded = encode(sentence)
print(f"encoded: {encoded}")


encoded: [29, 8, 13, 14, 18, 0, 20, 17, 18, 53, 0, 17, 4, 53, 30, 23, 19, 8, 13, 2, 19]


Character-level tokenization solves the out-of-vocabulary problem but introduces new challenges:

1. Longer sequences: because each word becomes many tokens, models need to process much longer inputs.
2. Weaker semantic representation: individual characters carry very little meaning, so models must learn relationships across many steps.
3. Higher computational cost: longer sequences lead to more tokens per input, which increases training and inference time.

To find a better balance between vocabulary size and sequence length, we move to subword-level tokenization next.

### 1.3 - Subword-level tokenization

Sub-word methods such as `Byte-Pair Encoding (BPE)`, `WordPiece`, and `SentencePiece` **learn** common groups of characters and merge them into tokens. For example, the word **unbelievable** might turn into three tokens: **["un", "believ", "able"]**. This approach strikes a balance between word-level and character-level methods and fix their limitations.

The BPE algorithm builds a vocabulary iteratively using the following process:
1. Start with individual characters (each character is a token).
2. Count all adjacent pairs of tokens in a large text corpus.
3. Merge the most frequent pair into a new token.

Repeat steps 2 and 3 until you reach the desired vocabulary size (for example, 50,000 tokens).

In the next cell, you will experiment with BPE in practice to see how it compresses text into meaningful subword units. Instead of implementing the algorithm from scratch, you will use a pretrained tokenizer, which was already trained on a large text corpus to build its vocabulary, such as the data used to train `GPT-2`. This allows you to see how BPE works in practice with a real, learned vocabulary.

In [10]:
from transformers import AutoTokenizer

# Step 1: Load a pretrained GPT-2 tokenizer from Hugging Face. 
# Refer to this to learn more: https://huggingface.co/docs/transformers/en/model_doc/gpt2

tokenizer = AutoTokenizer.from_pretrained("gpt2")



In [11]:
# Step 2: Use it to write encode and decode helper functions
def encode(text):
    return tokenizer.encode(text, add_special_tokens=False)    


def decode(ids):
    return tokenizer.decode(ids)

In [12]:
# 3. Inspect the tokens to see how BPE breaks words apart.
sample = "Unbelievable tokenization powers! 🚀"
ids = encode(sample)
print(f"encode sample: {encode(sample)}")
print(f"decode sample: {decode(ids)}")



encode sample: [3118, 6667, 11203, 540, 11241, 1634, 5635, 0, 12520, 248, 222]
decode sample: Unbelievable tokenization powers! 🚀


### 1.4 - TikToken

`tiktoken` is a fast, production-ready library for tokenization used by OpenAI models.
It is designed for efficiency and consistency with how OpenAI counts tokens in GPT models.

In this section, you will explore how different model families use different tokenizers. We will compare tokenizers used to train `GPT-2` and more powerful models such as `GPT-4`. By trying both, you will see how tokenization has evolved to handle more diverse text (including emojis, Unicode, and special characters) while remaining efficient.

In the next cell, you will use tiktoken to load these encodings and inspect how each one splits the same text. You may find reading this doc helpful: https://github.com/openai/tiktoken

In [13]:
import tiktoken

# Compare GPT-2 and GPT-4 tokenizers using tiktoken.

# Step 1: Load two tokenizers

tokenizerGpt2 = tiktoken.encoding_for_model("gpt-2")
tokenizerGpt4 = tiktoken.encoding_for_model("gpt-4o")

# Step 2: Encode the same sentence with both and observe how they differ
sentence = "The 🌟 star-programmer. *"
encoded2 = tokenizerGpt2.encode(sentence)
encoded4 = tokenizerGpt4.encode(sentence)

print(f":encoded2 : {encoded2}")
print(f":encoded4 : {encoded4}")


:encoded2 : [464, 12520, 234, 253, 3491, 12, 23065, 647, 13, 1635]
:encoded4 : [976, 130321, 253, 8253, 81630, 1159, 13, 425]


Try changing the input sentence and observe how different tokenizers behave.
Experiment with:
- Emojis, special characters, or punctuation
- Code snippets or structured text
- Non-English text (for example, Japanese, French, or Arabic)

If you are curious, you can also attempt to implement the BPE algorithm yourself using a small text corpus to see how token merges are learned in practice.

### 1.5 - Key Takeaways
- **Word-level**: simple and intuitive, but limited by large vocabularies and out-of-vocabulary issues
- **Character-level**: flexible and covers all text, but produces long sequences that are harder to model
- **Subword / BPE**: balances both worlds and is the default choice for most modern LLMs
- **TikToken**: a production-ready tokenizer used in OpenAI models, demonstrating how optimized subword vocabularies are applied in real systems

# 2. What is a Language Model?

At its core, a **language model (LM)** is just a *very large* mathematical function built from many neural-network layers.  
Given a sequence of tokens `[t₁, t₂, …, tₙ]`, it learns to output a probability for the next token `tₙ₊₁`.


Each layer performs basic mathematical operations such as matrix multiplication and attention. When hundreds of these layers are stacked together, the model learns complex patterns and statistical relationships in text. The final output is a vector of scores that represents how likely each possible token is to appear next. You can think of the entire model as one giant equation whose parameters were optimized during training to minimize prediction errors.

### 2.1 - A Single `Linear` Layer

Before jumping into Transformers, let's start with the simplest building block: a `Linear` layer.

A Linear layer computes `y = Wx + b`.

Where:  
  * `x` - input vector  
  * `W` - weight matrix (learned)  
  * `b` - bias vector (learned)

Although this operation looks simple, stacking many linear layers (along with nonlinear activation functions) allows neural networks to model highly complex relationships in data.

In the next cell, you will explore how a **Linear layer** works in practice by implementing one from scratch. You will define the weights and bias, then perform the matrix multiplication and addition manually to see what happens inside this layer. You may find the following links useful:
- https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html
- https://docs.pytorch.org/docs/stable/generated/torch.randn.html
- https://docs.pytorch.org/docs/stable/generated/torch.matmul.html

In [14]:
import torch
import torch.nn as nn

# Define a MyLinear PyTorch module and perform y = Wx + b.

class MyLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(MyLinear, self).__init__()
        # Initialize weights and bias as learnable parameters. 
        self.weights = nn.Parameter(torch.randn(out_features, in_features))  # random normal
        self.bias = nn.Parameter(torch.zeros(out_features))  # start with zeros


    def forward(self, x):
        # Matrix multiplication followed by bias addition
        # Linear transformation: y = xW^T + b
        return x @ self.weights.T + self.bias
        


lin = MyLinear(3, 2)
x = torch.tensor([1.0, -1.0, 0.5])
print("Input :", x)
print("Weights:", lin.weights)
print("Bias   :", lin.bias)
print("Output :", lin(x))

Input : tensor([ 1.0000, -1.0000,  0.5000])
Weights: Parameter containing:
tensor([[-1.7842, -0.0423, -0.0814],
        [-1.3449, -0.6983, -0.1688]], requires_grad=True)
Bias   : Parameter containing:
tensor([0., 0.], requires_grad=True)
Output : tensor([-1.7826, -0.7310], grad_fn=<AddBackward0>)


Next, you will use PyTorch's built-in nn.Linear module, which performs the same computation `(y = Wx + b)` but automatically handles parameter initialization, gradient tracking, and integration with the rest of a neural network. Comparing your manual implementation with this built-in version will help you understand what a linear layer does and how deep learning frameworks make these operations easier to use.

You may find this link useful:
- https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html

In [15]:
import torch.nn as nn, torch

# Create a linear layer using pytorch's nn.Linear
# Create a linear layer with 3 inputs and 2 outputs
lin = nn.Linear(in_features=3, out_features=2)

x = torch.tensor([1.0, -1.0, 0.5])
print("Input :", x)
print("Weights:", lin.weight)
print("Bias   :", lin.bias)
print("Output :", lin(x))


Input : tensor([ 1.0000, -1.0000,  0.5000])
Weights: Parameter containing:
tensor([[ 0.3462, -0.3046, -0.4105],
        [ 0.4087,  0.4355, -0.2716]], requires_grad=True)
Bias   : Parameter containing:
tensor([ 0.2975, -0.3702], requires_grad=True)
Output : tensor([ 0.7431, -0.5328], grad_fn=<ViewBackward0>)


### 2.2 - A `Transformer` Layer

Most LLMs are a **stack of identical Transformer blocks**. Each block fuses two main components:

| Step | What it does | Where it lives in code |
|------|--------------|------------------------|
| **Multi-Head Self-Attention** | Every token looks at every other token and decides *what matters*. | `block.attn` |
| **Feed-Forward Network (MLP)** | Re-mixes information token-by-token. | `block.mlp` |

In the next section, you will load `GPT-2` and inspect its first Transformer block to see these components in a real model. You will locate its layers, print their shapes and parameters, and understand how a block processes a batch of token embeddings.

In [16]:
import torch
from transformers import GPT2LMHeadModel

# Step 1: load the smallest GPT-2 model (124M parameters) using the Hugging Face transformers library.
# Refer to: https://huggingface.co/docs/transformers/en/model_doc/gpt2

model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

# Step 2: # Inspect the first Transformer block one by printing it.
block = model.transformer.h[0]
print(block)


GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


In this section, you will run a minimal forward pass through one GPT-2 block to understand how tokens are transformed inside the model.

In [17]:
# Step 1: Create a small dummy input with a sequence of 8 random token IDs.
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
dummy_input_ids = torch.randint(0, tokenizer.vocab_size, (1, 8))  # (batch_size=1, seq_len=8)


# Step 2: Convert token IDs into embeddings
# GPT-2 uses two embedding layers:
#   - wte (word token embeddings)
#   - wpe (positional embeddings)
# Add them together to form the initial hidden representation of your input tokens.
#   - wte (word token embeddings) - The word embeddings = what the words mean
token_embeddings = model.transformer.wte(dummy_input_ids)  # shape: [batch, seq_len, hidden_dim]
#   - wpe (positional embeddings) - The position embeddings = the order of words in the sentence
position_ids = torch.arange(0, dummy_input_ids.size(1)).unsqueeze(0)  # shape: [1, seq_len]
position_embeddings = model.transformer.wpe(position_ids)
# Add them to form the initial hidden states
hidden_states = token_embeddings + position_embeddings

# Step 3: Pass the embeddings through a single Transformer block
# This simulates one layer of computation in GPT-2.
 
output = model.transformer.h[0](hidden_states)

# Step 4: Inspect the result
# The output shape should be (batch_size, sequence_length, hidden_size)

print("Output shape:", output[0].shape)



Output shape: torch.Size([1, 8, 768])


### 2.3 - Inside GPT-2

GPT-2 is essentially a stack of identical Transformer blocks arranged in sequence.
Each block contains attention, feed-forward, and normalization layers that process token representations step by step.

In this section, you will print the modules inside the GPT-2 Transformer to see how these components are organized.
This will help you understand how the model scales from a single block to a full network of many layers working together.

In [18]:
# Print the name of all layers inside gpt.transformer. 
# You may find this helpful: https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.named_children
from transformers import GPT2LMHeadModel
gpt = GPT2LMHeadModel.from_pretrained("gpt2")
for name, module in gpt.transformer.named_children():
    print(name, ":", module)
print("over")


wte : Embedding(50257, 768)
wpe : Embedding(1024, 768)
drop : Dropout(p=0.1, inplace=False)
h : ModuleList(
  (0-11): 12 x GPT2Block(
    (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (attn): GPT2Attention(
      (c_attn): Conv1D(nf=2304, nx=768)
      (c_proj): Conv1D(nf=768, nx=768)
      (attn_dropout): Dropout(p=0.1, inplace=False)
      (resid_dropout): Dropout(p=0.1, inplace=False)
    )
    (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (mlp): GPT2MLP(
      (c_fc): Conv1D(nf=3072, nx=768)
      (c_proj): Conv1D(nf=768, nx=3072)
      (act): NewGELUActivation()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)
ln_f : LayerNorm((768,), eps=1e-05, elementwise_affine=True)
over


As you can see, the Transformer holds various modules, arranged from a list of blocks (`h`). The following table summarizes these modules:

| Step | What it does | Why it matters |
|------|--------------|----------------|
| **Token → Embedding** | Converts IDs to vectors | Gives the model a numeric “handle” on words |
| **Positional Encoding** | Adds “where am I?” info | Order matters in language |
| **Multi-Head Self-Attention** | Each token asks “which other tokens should I look at?” | Lets the model relate words across a sentence |
| **Feed-Forward Network** | Two stacked Linear layers with a non-linearity | Mixes information and adds depth |
| **LayerNorm & Residual** | Stabilize training and help gradients flow | Keeps very deep networks trainable |


### 2.4 LLM's output

When you pass a sequence of tokens through a language model, it produces a tensor of logits with shape
`(batch_size, seq_len, vocab_size)`.
Each position in the sequence receives a vector of scores representing how likely every possible token is to appear next. By applying a softmax function on the last dimension, these logits can be converted into probabilities that sum to 1.

In the next cell, you will feed an 8-token dummy sequence into GPT-2, print the shape of its logits, and display the five most likely next tokens predicted for the final position in the sequence.


In [19]:
import torch, torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# Step 1: Load GPT-2 model and its tokenizer
gpt = GPT2LMHeadModel.from_pretrained("gpt2")
gptTokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

In [20]:
# Step 2: Tokenize input text
text = "Hello my name"
input_ids = gptTokenizer.encode(text, return_tensors="pt")


In [21]:
# Step 3: Pass the input IDs to the model
with torch.no_grad():  # disables gradient tracking (we’re not training)
    outputs = gpt(input_ids)
    logits = outputs.logits

In [22]:
# Step 4: Predict the next token
# We take the logits from the final position, apply softmax to get probabilities,
# and then extract the top 5 most likely next tokens. You may find F.softmax and torch.topk helpful in your implementation.

last_token_logits = logits[0, -1, :]  # logits for the last position in the sequence
probs = F.softmax(last_token_logits, dim=-1)  # convert logits → probabilities
top_probs, top_indices = torch.topk(probs, k=5)  # get top 5 likely next tokens

for i in range(5):
    token = gptTokenizer.decode(top_indices[i])
    print(f"{token}: {top_probs[i].item():.4f}")

 is: 0.7773
,: 0.0373
's: 0.0332
 was: 0.0127
 and: 0.0076


### 2.5 - Key Takeaway

A language model is not a black box or something mysterious.
It is a large composition of simple, understandable layers such as linear layers, attention, and normalization, trained together to predict the next token in a sequence.

By learning this next-token prediction task at scale, the model gradually develops an internal understanding of language structure, meaning, and context, which allows it to generate coherent and relevant text.

# 3 - Text Generation (Decoding)
Once a language model has been trained to predict token probabilities, we can use it to generate text.
This process is called text generation or decoding.

At each step, the model outputs a probability distribution over possible next tokens.
A decoding algorithm then selects one token based on that distribution, appends it to the sequence, and repeats the process to build text word by word. Different decoding strategies control how the model chooses the next token and how creative or deterministic the output will be. For example:
- **Greedy** decoding: always pick the token with the highest probability. Simple and consistent, but often repetitive.
- **Top-k** or **Nucleus** (top-p) sampling: randomly sample from the top few likely tokens to add variety.
- Beam search: explores multiple candidate continuations and keeps the best overall sequence.

Note: `Temperature` adjusts randomness in sampling. Higher values make outputs more diverse, while lower values make them more focused and deterministic.

### 3.1 - Greedy decoding
In this section, you will use GPT-2 and Hugging Face's built-in generate method to produce text using the greedy decoding strategy.

In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Step 1. Load GPT-2 model and tokenizer.
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

 

     # Ensure pad_token_id is set for open-ended generation if not already.
    # If pad_token is the same as eos_token, explicitly set pad_token_id to eos_token_id
    # for the generation process if it's not already handled by the model's config.
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    # print(f"Setting pad_token_id to eos_token_id: {tokenizer.eos_token_id}")


# Step 2. Implement a text generation function using HuggingFace's generate method.
def generate(model, tokenizer, prompt, max_new_tokens=128):
     # Encode the prompt into input IDs
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    # Generate text using the model's generate method
    # The 'max_new_tokens' argument controls the length of the generated output
    # 'pad_token_id' is set to the tokenizer's eos_token_id to handle padding during generation
    generated_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else model.config.eos_token_id,
        # Add other generation parameters as needed, e.g., do_sample, top_k, top_p, temperature
    )

    # Decode the generated token IDs back into text
    # 'skip_special_tokens=True' ensures that special tokens like [CLS], [SEP], etc., are not included in the output
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    return generated_text

In [24]:
tests=["Once upon a time","What is 2+2?", "Suggest a party theme."]
for prompt in tests:
    print(f"\n GPT-2 | Greedy")
    print(generate(model, tokenizer, prompt, 80))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



 GPT-2 | Greedy
Once upon a time, the world was a place of great beauty and great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and

 GPT-2 | Greedy
What is 2+2?

2+2 is the number of times you can use a spell to cast a spell.

2+2 is the number of times you can use a spell to cast a spell.

2+2 is the number of times you can use a spell to cast a spell.

2+2 is the number of times you can use a spell to cast a spell.

 GPT-2 | Greedy
Suggest a party theme.

The party theme is a simple, simple, and fun way to get your friends to join you.

The party theme is a simple, simple, and fun way to get your friends to join you. The party theme is a simple, simple, and fun way to get your friends to join you. The party theme is a simple, simple, and f

Naively selecting the single most probable token at each step (known as greedy decoding) often leads to poor results in practice:
- Repetition loops: phrases like “The cat is is is…”
- Short-sighted choices: the most likely token right now might lead to incoherent text later

These issues are why more advanced decoding methods such as top-k and nucleus sampling are commonly used to make model outputs more diverse and natural.

### 3.2 - Top-k and top-p sampling
The generate function you implemented earlier can easily be extended to use different decoding strategies.

In this section, you will reimplement the same function but adapt it to support Top-k and Top-p (nucleus) sampling. These methods introduce controlled randomness, allowing the model to explore multiple plausible continuations instead of always choosing the single most likely next token.

In [25]:
# Implement `generate` to support 3 strategies: greedy, top_k, and top_o
# You may find this link helpful: https://huggingface.co/docs/transformers/en/main_classes/text_generation

 
def generate(model, tokenizer, prompt, strategy="top_p", max_new_tokens=128,temperature=1.5, top_k=50, top_p=0.9):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            next_token_logits = outputs.logits[:, -1, :]  # Logits for last token

            # Apply temperature scaling
            if temperature != 1.0:
                next_token_logits = next_token_logits / temperature

            # Sampling strategy
            if strategy == "greedy":
                next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            elif strategy == "top_k":
                # Keep only top_k logits
                top_k = min(top_k, next_token_logits.size(-1))  # safety check
                top_k_logits, top_k_indices = torch.topk(next_token_logits, k=top_k, dim=-1)
                probs = F.softmax(top_k_logits, dim=-1)
                sampled_idx = torch.multinomial(probs, num_samples=1)
                next_token = top_k_indices.gather(-1, sampled_idx)

            elif strategy == "top_p":
                # Sort logits by probability
                sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True, dim=-1)
                probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(probs, dim=-1)

                # Mask tokens with cumulative probability above top_p
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
                sorted_indices_to_remove[:, 0] = 0

                # Apply the mask
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                next_token_logits[indices_to_remove] = float("-inf")

                # Sample from filtered distribution
                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            
            else:
                raise ValueError(f"Unknown strategy: {strategy}")

            # Append the predicted token
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
            ], dim=-1)

            # Stop at EOS
            if tokenizer.eos_token_id is not None and next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [26]:
# Step 1. Load GPT-2 model and tokenizer.

tests=["Rajdeep left for work","I think Snoopy is", "Suggest a party theme."]
for prompt in tests:
    print(f"\n GPT-2 | Top-p")
    print(generate(model, tokenizer, prompt, "top_p", 40))


 GPT-2 | Top-p
Rajdeep left for work on posts 5(Oil Asset Raising Royalties — : road machines grad isn'twromochc/MiSw]] There will be a further credibility demonstration and after many |McShuk cre

 GPT-2 | Top-p
I think Snoopy is innocent these days of reflection please like it's ok WEBUI person th— Cam Cly50 pen famect gnot off Ohio nUKNG Lindsay Klein ShareHey Lake RdKTö Noveltyriver

 GPT-2 | Top-p
Suggest a party theme. ConsolegenDiskXbox mentions Minifig support on this market from Gabe Gall and Gabriel Garrell via Gamergate Support OUYAViaForm to exPanulator for instructions see Wiki really supporting Contact Kagavy 


### 3.3 - Try It Yourself

Now it’s time to experiment with text generation. Replace the sample prompts with your own prompts or adjust the decoding strategy.
You can experiment with:
- strategy: "greedy", "beam", "top_k", "top_p"
- temperature: values between 0.2 and 2.0
- k or p: thresholds that control sampling diversity

Try generating the same prompt with `greedy` and `top_p` (for example, 0.9). Notice how even small temperature changes can make the output more focused or more free-form.




# 4 - Completion vs. Instruction-tuned LLMs

So far, we have used `GPT-2` to generate text from a given input prompt. However, `GPT-2` is just a completion model. It simply continues the provided text without understanding it as a task or question. It is not designed to engage in dialogue or follow instructions.

In contrast, instruction-tuned LLMs (such as `Qwen-Chat`) undergo an additional post-training stage after base pre-training. This process fine-tunes the model to behave helpfully and safely when interacting with users. Because of this extra stage, instruction-tuned models can:

- Interpret prompts as requests rather than just text to continue
- Stay in conversation mode, answering questions and following steps
- Handle refusals and safety boundaries appropriately
- Maintain a consistent helpful persona, rather than drifting into storytelling

### 4.1 - `Qwen/Qwen3-0.6B` vs. `GPT2`

In the next cell, you will feed the same prompt to two different models:

- GPT-2 (completion-only): continues the text in the same writing style
- Qwen/Qwen3-0.6B (instruction-tuned): interprets the input as an instruction and responds helpfully

Comparing the two outputs will make the difference between completion and instruction-tuned behavior clear.



In [27]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load both GPT-2 and Qwen models using HuggingFace `.from_pretrained` method.
# Load GPT-2 model
modelGpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizerGpt2 = AutoTokenizer.from_pretrained("gpt2")

# Load Qwen model (use a valid model name, for example Qwen 1.5B)
# Load Qwen3-0.6B
device = "cuda" if torch.cuda.is_available() else "cpu"
# Load Qwen3-0.6B model and tokenizer
# Load tokenizer first
tokenizer_qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

# Some Qwen tokenizers have no pad_token by default, so fix it
if tokenizer_qwen.pad_token is None:
    tokenizer_qwen.pad_token = tokenizer_qwen.eos_token

# Load model
model_qwen = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)


We have now downloaded two small checkpoints: GPT-2 (124M parameters) and Qwen3-0.6B (600M parameters). If the previous cell took some time to run, that was mainly due to model download speed. The models will be cached locally, so future runs will be faster.

Next, we will generate text using our generate function with both models and the same prompt to directly compare how a completion-only model (GPT-2) behaves differently from an instruction-tuned model (Qwen).

In [28]:

tests=[("Once upon a time", "top_p"),("What is 2+2?", "top_k"),("Suggest a party theme.", "top_p")]

for text, strategy in tests:
    print(f"\nGPT-2 | {strategy}")
    print(generate(modelGpt2, tokenizerGpt2, text, strategy, 40))

    print(f"\nQwen-3 | {strategy}")
    print(generate(model_qwen, tokenizer_qwen, text, strategy, 40))



GPT-2 | top_p
Once upon a time castle thus fit truly In A Melman Sepi knew arms opened flaming awesome begineth Seventeen spells aloha bristordon Versarians Khui Slatar these displays of ki Various have magic definition rekan

Qwen-3 | top_p
Once upon a time—a Thursday village laid Shield during an afternoon before EF School freshman negotiations described validating and guiding front home_schooles..."


something sterile recovery landblogs

 Tylen des/kko lumos might pour rage

GPT-2 | top_k
What is 2+2?

It is important first of all where you start

1-20 minutes of practice,

2 sessions a week each week

6 hours daily between practices 1 (not just in

Qwen-3 | top_k
What is 2+2? Well if you're already doing two-digit math, like "2 dollars plus two dollars," each person is supposed to pay some amount so that the two would meet to spend something.

Let me explain how

GPT-2 | top_p
Suggest a party theme. Hold goal and coord yield clips outstretched from each plot green placement each 

# 5. (Optional) A Small Interactive LLM Playground
This section is optional. You do not need to implement it to complete the project. It is meant purely for exploration and will not significantly affect your core AI engineering skills.

If you are curious, you can build a simple interactive playground to experiment with text generation. You can:
- Create input widgets for the prompt, model selection, decoding strategy, and temperature
- Use Hugging Face's generate method to produce text based on the selected settings
- Display the model's response directly in the notebook output

You may find following links helpful:
- https://ipywidgets.readthedocs.io/en/latest/
- https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html

In [11]:
import ipywidgets as widgets
from IPython.display import display, Markdown
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

# Steps to implement:
# Determine device + dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# 1. Load models and tokenizers (GPT-2 and Qwen).
modelGpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizerGpt2 = AutoTokenizer.from_pretrained("gpt2")

modelQwen = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    dtype = torch.float16 if device == "cuda" else torch.float32,
    trust_remote_code=True
)

tokenizer_qwen = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", 
    trust_remote_code=True
)# 2. Define a helper function to generate text with different decoding strategies.

def generate(model, tokenizer, prompt, strategy="top_p", max_new_tokens=128,temperature=1.5, top_k=50, top_p=0.9):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            next_token_logits = outputs.logits[:, -1, :]  # Logits for last token

            # Apply temperature scaling
            if temperature != 1.0:
                next_token_logits = next_token_logits / temperature

            # Sampling strategy
            if strategy == "greedy":
                next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            elif strategy == "top_k":
                # Keep only top_k logits
                top_k = min(top_k, next_token_logits.size(-1))  # safety check
                top_k_logits, top_k_indices = torch.topk(next_token_logits, k=top_k, dim=-1)
                probs = F.softmax(top_k_logits, dim=-1)
                sampled_idx = torch.multinomial(probs, num_samples=1)
                next_token = top_k_indices.gather(-1, sampled_idx)

            elif strategy == "top_p":
                # Sort logits by probability
                sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True, dim=-1)
                probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(probs, dim=-1)

                # Mask tokens with cumulative probability above top_p
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
                sorted_indices_to_remove[:, 0] = 0

                # Apply the mask
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                next_token_logits[indices_to_remove] = float("-inf")

                # Sample from filtered distribution
                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            
            else:
                raise ValueError(f"Unknown strategy: {strategy}")

            # Append the predicted token
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
            ], dim=-1)

            # Stop at EOS
            if tokenizer.eos_token_id is not None and next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

# 3. Create interactive UI elements (prompt box, model selector, strategy selector, temperature slider).

# UI elements
prompt_box = widgets.Textarea(description="Prompt:", layout=widgets.Layout(width="600px", height="120px"))
model_selector = widgets.Dropdown(options=["GPT-2", "Qwen"], description="Model:")
strategy_selector = widgets.Dropdown(options=["greedy", "top_k", "top_p"], description="Strategy:")
temperature_slider = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description="Temp:")

# 4. Add a button to trigger text generation.
# 5. Define the button’s behavior.
# 6. Display the full UI for the playground.


generate_button = widgets.Button(description="Generate", button_style="success")
output_area = widgets.Output()

def on_click(_):
    output_area.clear_output()
    with output_area:
        if model_selector.value == "GPT-2":
            model, tokenizer = modelGpt2, tokenizerGpt2
        else:
            model, tokenizer = modelQwen, tokenizer_qwen

        text = generate(
            model, tokenizer, 
            prompt_box.value,
            strategy=strategy_selector.value,
            temperature=temperature_slider.value
        )
        display(Markdown(f"**Output:**\n\n{text}"))

generate_button.on_click(on_click)

display(prompt_box, model_selector, strategy_selector, temperature_slider, generate_button, output_area)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Dropdown(description='Model:', options=('GPT-2', 'Qwen'), value='GPT-2')

Dropdown(description='Strategy:', options=('greedy', 'top_k', 'top_p'), value='greedy')

FloatSlider(value=1.0, description='Temp:', max=2.0, min=0.1)

Button(button_style='success', description='Generate', style=ButtonStyle())

Output()


## 🎉 Congratulations!

You've just learned, explored, and inspected a real **LLM**. In one project you:
* Learned how **tokenization** works in practice
* Used `tiktoken` library to load and experiment with most advanced tokenizers.
* Explored LLM architecture and inspected GPT2 blocks and layers
* Learned decoding strategies and used `top-p` to generate text from GPT2
* Loaded a powerful chat model, `Qwen3-0.6B` and generated text
* Built an LLM playground


👏 **Great job!** Take a moment to celebrate. You now have a working mental model of how LLMs work. The skills you used here power most LLMs you see everywhere.
